# Heike Monogatari: Tag Analysis

This notebook analyzes the `heike.sadler1918.xml` file to count the occurrences of `<persName>` (people) and `<placeName>` (places).

It provides:
1.  **Overall Totals**: The total count of each tag across the entire document.
2.  **Chapter Subtotals**: A table showing the counts for each chapter.
3.  **Normalized Frequency**: The chapter subtotals are normalized by the chapter's size (word count) to show the *density* of tags, presented as "Tags per 1000 Words".

In [14]:
import xml.etree.ElementTree as ET
import pandas as pd
import re
import os

print("Libraries imported successfully.")

Libraries imported successfully.


In [15]:
# --- Configuration ---

# Define the file name to analyze
XML_FILE = "/Users/gcrane/github/GRC_misc/heike.sadler1918.xml"

# Define the TEI namespace. This is crucial for parsing TEI XML.
# Without this, the parser won't find any tags.
NAMESPACE = {'tei': 'http://www.tei-c.org/ns/1.0'}

# Define the XML namespace for attributes like 'xml:id'
XML_NS = '{http://www.w3.org/XML/1998/namespace}'

# Tags to count
PERSON_TAG = 'tei:persName'
PLACE_TAG = 'tei:placeName'
CHAPTER_TAG = 'tei:div' # We'll filter this by attribute

In [16]:
def count_words(element):
    """
    Recursively gets all text from an element and its children,
    then returns a simple word count.
    """
    if element is None:
        return 0
    
    # itertext() gets text from the current element and all sub-elements
    all_text = " ".join(element.itertext())
    
    # Split by whitespace to get a simple word count
    words = all_text.split()
    return len(words)

print("Helper function `count_words` defined.")

Helper function `count_words` defined.


In [17]:
# --- Main Analysis ---

# Initialize lists and counters
chapter_data = []
total_people = 0
total_places = 0
total_words = 0

# Check if file exists
if not os.path.exists(XML_FILE):
    print(f"Error: File not found at {XML_FILE}")
    print("Please make sure the XML file is in the same directory as this notebook.")
else:
    # Parse the XML file
    tree = ET.parse(XML_FILE)
    root = tree.getroot()

    # Find all chapters
    # We look for <div type="textpart" subtype="chapter">
    chapters = root.findall(f'.//{CHAPTER_TAG}[@type="textpart"][@subtype="chapter"]', NAMESPACE)
    
    if not chapters:
        # Fallback if the first query finds nothing (e.g., if only subtype="volume" exists)
        print("No <div type='textpart' subtype='chapter'> found.")
        print("Trying to find <div type='textpart' subtype='volume'> as fallback...")
        chapters = root.findall(f'.//{CHAPTER_TAG}[@type="textpart"][@subtype="volume"]', NAMESPACE)
        if not chapters:
            print("No chapters or volumes found. Please check XML structure and tags.")
    
    print(f"Found {len(chapters)} chapters/volumes to analyze.")

    # Loop through each chapter and extract data
    for chapter in chapters:
        # Get chapter ID (which uses the 'xml' namespace)
        chapter_id = chapter.get(f'{XML_NS}id', 'Unknown')
        
        # Count tags *within* this chapter
        # The './/' prefix is important to search all descendants of the chapter
        people_count = len(chapter.findall(f'.//{PERSON_TAG}', NAMESPACE))
        place_count = len(chapter.findall(f'.//{PLACE_TAG}', NAMESPACE))
        
        # Get chapter size
        word_count = count_words(chapter)
        
        # Calculate totals for this chapter
        total_tags = people_count + place_count
        
        # Calculate normalized frequency (tags per 1000 words)
        if word_count > 0:
            normalized_freq = (total_tags / word_count) * 1000
        else:
            normalized_freq = 0
        
        # Add to our chapter list
        chapter_data.append({
            "Chapter ID": chapter_id,
            "People": people_count,
            "Places": place_count,
            "Total Tags": total_tags,
            "Word Count": word_count,
            "Tags per 1000 Words": round(normalized_freq, 2)
        })
        
        # Add to overall totals
        total_people += people_count
        total_places += place_count
        total_words += word_count

    print("Analysis complete.")

Found 189 chapters/volumes to analyze.
Analysis complete.


## Chapter-by-Chapter Results

The table below shows the raw counts for `<persName>` and `<placeName>` in each chapter, the chapter's total word count, and the normalized frequency of "Tags per 1000 Words".

This normalized score helps compare tag density between chapters of different lengths.

In [18]:
# Create and display the DataFrame
if not chapter_data:
    print("No data to display. The analysis might have failed to find any chapters.")
else:
    df = pd.DataFrame(chapter_data)
    
    # Set Chapter ID as the index for better readability
    df.set_index("Chapter ID", inplace=True)
    
    # Display the full DataFrame
    with pd.option_context('display.max_rows', None, 'display.max_columns', None):
        print(df)

            People  Places  Total Tags  Word Count  Tags per 1000 Words
Chapter ID                                                             
ch1             34       6          40         510                78.43
ch2             38       8          46        1571                29.28
ch3             36      10          46         912                50.44
ch4              6       5          11         441                24.94
ch5             29       8          37         458                80.79
ch6             89      10          99        4244                23.33
ch7             32      14          46        1266                36.33
ch8             21      22          43         666                64.56
ch9             43      21          64        1106                57.87
ch10            52      22          74        1261                58.68
ch11            80      13          93        1474                63.09
ch12            80      33         113        1660              

In [21]:
# --- Main Analysis Script ---

all_people_names = []
all_place_names = []

# Check if file exists
if not os.path.exists(XML_FILE):
    print(f"Error: File not found at {XML_FILE}")
    print("Please make sure the XML file is in the same directory as this notebook.")
else:
    # Parse the XML file
    tree = ET.parse(XML_FILE)
    root = tree.getroot()

    # --- Find all PEOPLE ---
    # We use .// to find all tags anywhere in the document
    for person_tag in root.findall(f'.//{PERSON_TAG}', NAMESPACE):
        # Check if the tag has text before adding
        if person_tag.text:
            name = person_tag.text.strip()
            if name: # Ensure it's not just whitespace
                all_people_names.append(name)

    # --- Find all PLACES ---
    for place_tag in root.findall(f'.//{PLACE_TAG}', NAMESPACE):
        if place_tag.text:
            place = place_tag.text.strip()
            if place:
                all_place_names.append(place)

    print(f"Analysis complete.")
    print(f"Found {len(all_people_names)} total references to people (across {len(set(all_people_names))} unique names).")
    print(f"Found {len(all_place_names)} total references to places (across {len(set(all_place_names))} unique names).")

Analysis complete.
Found 8416 total references to people (across 3542 unique names).
Found 3710 total references to places (across 1244 unique names).


## Overall Book Totals

In [22]:
# Calculate and print overall totals
if total_words > 0:
    overall_total_tags = total_people + total_places
    overall_normalized = (overall_total_tags / total_words) * 1000

    print(f"--- Overall Statistics for '{XML_FILE}' ---")
    print(f"Total People Tags (<persName>): {total_people}")
    print(f"Total Place Tags (<placeName>):  {total_places}")
    print("---------------------------------------------")
    print(f"Overall Total Tags:             {overall_total_tags}")
    print(f"Overall Word Count:             {total_words}")
    print(f"Overall Normalized Frequency:   {overall_normalized:.2f} tags per 1000 words")
else:
    print("No words or tags were counted. Cannot calculate overall totals.")

--- Overall Statistics for '/Users/gcrane/github/GRC_misc/heike.sadler1918.xml' ---
Total People Tags (<persName>): 7850
Total Place Tags (<placeName>):  3487
---------------------------------------------
Overall Total Tags:             11337
Overall Word Count:             221218
Overall Normalized Frequency:   51.25 tags per 1000 words


In [23]:
# Use collections.Counter to get frequencies
people_frequency = Counter(all_people_names)

# Convert to a Pandas DataFrame for nice formatting
people_df = pd.DataFrame(people_frequency.most_common(20), columns=['Person', 'Count'])
people_df.set_index('Person', inplace=True)

print(people_df)

NameError: name 'Counter' is not defined

In [20]:
# Use collections.Counter to get frequencies
place_frequency = Counter(all_place_names)

# Convert to a Pandas DataFrame for nice formatting
place_df = pd.DataFrame(place_frequency.most_common(20), columns=['Place', 'Count'])
place_df.set_index('Place', inplace=True)

print(place_df)

NameError: name 'Counter' is not defined